# GOP (Goodness of Pronunciation) Prototyping

This notebook experiments with **Forced Alignment** and **Goodness-of-Pronunciation** scoring
using the `torchaudio.pipelines.MMS_FA` (Wav2Vec2-based) model.

## Roadmap

1. **Forced Alignment** – Align a reference transcript to audio at the phoneme level
2. **Log-Probability Extraction** – Extract per-frame log-posteriors from the acoustic model
3. **GOP Scoring** – Compute `GOP(p) = (1/d) * Σ log P(p | o_t)` for each aligned phoneme
4. **Integration** – Port the working pipeline to `backend/app/ai_pipeline/pronunciation.py`

---
## 1. Environment Setup

In [ ]:
import torch
import torchaudio

print(f"PyTorch version:  {torch.__version__}")
print(f"Torchaudio version: {torchaudio.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:              {torch.cuda.get_device_name(0)}")

---
## 2. Load the MMS Forced-Alignment Model

We use `torchaudio.pipelines.MMS_FA` which provides a Wav2Vec2 model
fine-tuned for multilingual forced alignment. It outputs per-frame
log-probabilities over a phoneme inventory (IPA-based).

In [ ]:
# Load the MMS Forced Alignment pipeline
bundle = torchaudio.pipelines.MMS_FA

model = bundle.get_model()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

# Inspect the label/token dictionary
LABELS = bundle.get_labels()
DICTIONARY = bundle.get_dict()

print(f"Model loaded on: {device}")
print(f"Number of labels: {len(LABELS)}")
print(f"Labels (first 20): {LABELS[:20]}")

---
## 3. Load a Sample Audio File

Replace the path below with any `.wav` file for testing.
The model expects **16 kHz mono** audio.

In [ ]:
# --- Replace with your own test audio ---
AUDIO_PATH = "test_audio.wav"  # TODO: provide a real .wav file
TRANSCRIPT = "hello world"     # Reference transcript

# Load and resample to 16 kHz
waveform, sample_rate = torchaudio.load(AUDIO_PATH)
if sample_rate != bundle.sample_rate:
    waveform = torchaudio.functional.resample(waveform, sample_rate, bundle.sample_rate)

# Ensure mono
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)

print(f"Waveform shape: {waveform.shape}")
print(f"Duration: {waveform.shape[1] / bundle.sample_rate:.2f} s")

---
## 4. Run Forced Alignment

We tokenise the transcript into the model's label set, run a
forward pass to get per-frame log-probabilities, and use
`torchaudio.functional.forced_align` (CTC-based) to find the
optimal alignment path.

In [ ]:
def tokenise_transcript(transcript: str, dictionary: dict) -> list[int]:
    """
    Convert a transcript string to a list of token IDs using the
    MMS_FA label dictionary.
    
    Inserts a word-boundary token (pipe '|') between words.
    Unknown characters are skipped with a warning.
    """
    tokens = []
    for i, word in enumerate(transcript.lower().split()):
        if i > 0:
            # Insert word boundary
            if "|" in dictionary:
                tokens.append(dictionary["|"])
        for char in word:
            if char in dictionary:
                tokens.append(dictionary[char])
            else:
                print(f"Warning: character '{char}' not in dictionary, skipping.")
    return tokens


# Tokenise
token_ids = tokenise_transcript(TRANSCRIPT, DICTIONARY)
print(f"Transcript: '{TRANSCRIPT}'")
print(f"Token IDs:  {token_ids}")
print(f"Tokens:     {[LABELS[t] for t in token_ids]}")

In [ ]:
# Run the model forward pass to get per-frame emissions (log-probs)
with torch.no_grad():
    emission, _ = model(waveform.to(device))

print(f"Emission shape: {emission.shape}")  # [1, T, C]
print(f"  T (frames) = {emission.shape[1]}")
print(f"  C (labels) = {emission.shape[2]}")

In [ ]:
# Run CTC forced alignment
targets = torch.tensor([token_ids], dtype=torch.int32, device=device)
input_lengths = torch.tensor([emission.shape[1]], device=device)
target_lengths = torch.tensor([len(token_ids)], device=device)

aligned_tokens, alignment_scores = torchaudio.functional.forced_align(
    emission,
    targets,
    input_lengths,
    target_lengths,
    blank=0,  # CTC blank token is typically index 0
)

print(f"Aligned tokens shape: {aligned_tokens.shape}")
print(f"Alignment scores shape: {alignment_scores.shape}")

---
## 5. Extract Phoneme-Level Timestamps

Group consecutive aligned frames into phoneme segments with
start/end timestamps.

In [ ]:
def extract_segments(
    aligned_tokens: torch.Tensor,
    alignment_scores: torch.Tensor,
    labels: list[str],
    sample_rate: int,
    frames_per_step: int = 320,  # typical for Wav2Vec2 @ 16kHz
) -> list[dict]:
    """
    Convert a CTC alignment path into a list of phoneme segments.
    
    Each segment contains:
      - phoneme: str (label)
      - start: float (seconds)
      - end: float (seconds)
      - score: float (mean log-probability for this segment)
    """
    tokens = aligned_tokens[0].cpu().tolist()
    scores = alignment_scores[0].cpu().tolist()
    
    segments = []
    current_token = None
    start_frame = 0
    frame_scores = []
    
    for i, (tok, sc) in enumerate(zip(tokens, scores)):
        if tok == 0:  # blank
            if current_token is not None:
                segments.append({
                    "phoneme": labels[current_token],
                    "start": start_frame * frames_per_step / sample_rate,
                    "end": i * frames_per_step / sample_rate,
                    "score": sum(frame_scores) / max(len(frame_scores), 1),
                })
                current_token = None
                frame_scores = []
        elif tok != current_token:
            if current_token is not None:
                segments.append({
                    "phoneme": labels[current_token],
                    "start": start_frame * frames_per_step / sample_rate,
                    "end": i * frames_per_step / sample_rate,
                    "score": sum(frame_scores) / max(len(frame_scores), 1),
                })
            current_token = tok
            start_frame = i
            frame_scores = [sc]
        else:
            frame_scores.append(sc)
    
    # Handle last segment
    if current_token is not None:
        segments.append({
            "phoneme": labels[current_token],
            "start": start_frame * frames_per_step / sample_rate,
            "end": len(tokens) * frames_per_step / sample_rate,
            "score": sum(frame_scores) / max(len(frame_scores), 1),
        })
    
    return segments


segments = extract_segments(
    aligned_tokens,
    alignment_scores,
    LABELS,
    bundle.sample_rate,
)

print(f"\nFound {len(segments)} phoneme segments:\n")
for seg in segments:
    print(f"  {seg['phoneme']:>4s}  {seg['start']:6.3f}s – {seg['end']:6.3f}s  (score: {seg['score']:.4f})")

---
## 6. Next Steps: GOP Scoring

With the forced alignment above, we now have phoneme-level boundaries.
The next step is to compute **Goodness of Pronunciation (GOP)** for each phoneme:

$$
\text{GOP}(p) = \frac{1}{d} \sum_{t \in \text{segment}} \log P(p \mid o_t)
$$

where:
- $p$ is the canonical (expected) phoneme
- $o_t$ is the acoustic observation at frame $t$
- $d$ is the duration (number of frames) of the phoneme segment
- $P(p \mid o_t)$ is the posterior probability from the acoustic model

### Implementation Plan

1. **Extract per-frame log-posteriors** from the `emission` tensor for each aligned segment
2. **Look up** the log-probability of the expected phoneme at each frame within the segment
3. **Average** over the segment duration to get the GOP score
4. **Normalise** scores to a 0–100 scale for the frontend
5. **Port** the working pipeline to `backend/app/ai_pipeline/pronunciation.py`

The `alignment_scores` returned by `forced_align` already give us per-frame
log-probabilities along the alignment path, so the `score` field in our
segments above is effectively a raw GOP value. In the next iteration we'll
refine this by comparing against all competing phonemes.

In [ ]:
# Placeholder: compute a simple GOP score from the alignment scores
# This is a first approximation – the real GOP should compare against
# competing phonemes (see markdown cell above for the formula).

import numpy as np

def compute_simple_gop(segments: list[dict], scale_max: float = 100.0) -> list[dict]:
    """
    Compute a simplified GOP score normalised to [0, scale_max].
    
    Uses the mean log-probability from forced alignment as a proxy.
    Higher (less negative) log-probs → better pronunciation.
    """
    raw_scores = [seg["score"] for seg in segments if seg["phoneme"] != "|"]
    
    if not raw_scores:
        return segments
    
    # Normalise: map [min_score, 0] → [0, scale_max]
    min_score = min(raw_scores)
    score_range = abs(min_score) if min_score < 0 else 1.0
    
    for seg in segments:
        if seg["phoneme"] == "|":
            seg["gop_score"] = None
        else:
            normalised = (seg["score"] - min_score) / score_range * scale_max
            seg["gop_score"] = round(min(normalised, scale_max), 1)
    
    return segments


scored_segments = compute_simple_gop(segments)

print("Phoneme-level GOP scores:\n")
for seg in scored_segments:
    gop = seg.get("gop_score")
    if gop is not None:
        print(f"  {seg['phoneme']:>4s}  GOP: {gop:5.1f}/100")